# 01 - Setup Lakebase PostgreSQL Instance

This notebook creates a Lakebase PostgreSQL instance for the Personal Expense Tracker.

**Features:**
- Create PostgreSQL instance using Databricks SDK
- Configure instance for development
- Retrieve connection details
- Enable SSL connection
- Error handling and validation


## Install Required Libraries


In [ ]:
%pip install databricks-sdk psycopg2-binary sqlalchemy --quiet
dbutils.library.restartPython()


## Import Libraries


In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import DatabaseInfo, ProvisioningState
import time
import json
import os


## Configuration


In [ ]:
# Instance configuration
INSTANCE_NAME = "expense-tracker-lakebase"
DATABASE_NAME = "expense_tracker_db"
INSTANCE_TYPE = "SMALL"  # Small instance for development
STORAGE_SIZE_GB = 100  # 100GB storage

print(f"Instance Name: {INSTANCE_NAME}")
print(f"Database Name: {DATABASE_NAME}")
print(f"Instance Type: {INSTANCE_TYPE}")
print(f"Storage Size: {STORAGE_SIZE_GB}GB")


## Initialize Databricks Workspace Client


In [ ]:
try:
    # Initialize workspace client
    w = WorkspaceClient()
    print("✓ Successfully initialized Databricks Workspace Client")
    print(f"✓ Workspace URL: {w.config.host}")
except Exception as e:
    print(f"✗ Failed to initialize workspace client: {str(e)}")
    raise


## Check if Database Already Exists


In [ ]:
def check_database_exists(database_name):
    """Check if a Lakebase database already exists"""
    try:
        databases = w.database.list()
        for db in databases:
            if db.name == database_name:
                return True, db
        return False, None
    except Exception as e:
        print(f"Warning: Could not list databases: {str(e)}")
        return False, None

exists, existing_db = check_database_exists(DATABASE_NAME)
if exists:
    print(f"✓ Database '{DATABASE_NAME}' already exists")
    print(f"  Database ID: {existing_db.id}")
    print(f"  State: {existing_db.state}")
else:
    print(f"✓ Database '{DATABASE_NAME}' does not exist - will create new database")


## Create Lakebase PostgreSQL Database


In [ ]:
def create_lakebase_database():
    """Create a Lakebase PostgreSQL database"""
    try:
        print(f"Creating Lakebase database '{DATABASE_NAME}'...")
        
        # Create database using SDK
        database = w.database.create(
            name=DATABASE_NAME,
            instance_type=INSTANCE_TYPE,
            description="Personal Expense Tracker Database"
        )
        
        print(f"✓ Database creation initiated")
        print(f"  Database ID: {database.id}")
        print(f"  Database Name: {database.name}")
        
        return database
        
    except Exception as e:
        print(f"✗ Failed to create database: {str(e)}")
        raise

# Create database if it doesn't exist
if not exists:
    database = create_lakebase_database()
else:
    database = existing_db
    print("Using existing database")


## Wait for Database to be Ready


In [ ]:
def wait_for_database_ready(database_id, timeout_minutes=30):
    """Wait for the database to be in READY state"""
    print(f"\nWaiting for database to be ready (timeout: {timeout_minutes} minutes)...")
    
    start_time = time.time()
    timeout_seconds = timeout_minutes * 60
    
    while True:
        try:
            # Get database status
            current_db = w.database.get(id=database_id)
            state = current_db.state
            
            elapsed_time = int(time.time() - start_time)
            print(f"  [{elapsed_time}s] Current state: {state}")
            
            if state == ProvisioningState.ACTIVE:
                print(f"\n✓ Database is ACTIVE! (took {elapsed_time}s)")
                return current_db
            
            elif state == ProvisioningState.FAILED:
                raise Exception(f"Database provisioning failed")
            
            elif state in [ProvisioningState.PROVISIONING]:
                # Check timeout
                if elapsed_time > timeout_seconds:
                    raise TimeoutError(f"Database creation timed out after {timeout_minutes} minutes")
                
                # Wait before checking again
                time.sleep(30)
            else:
                print(f"  Unexpected state: {state}")
                time.sleep(30)
                
        except Exception as e:
            print(f"\n✗ Error while waiting: {str(e)}")
            raise

# Wait for database to be ready
ready_database = wait_for_database_ready(database.id)


## Retrieve Connection Details


In [ ]:
def get_connection_details(database_id):
    """Retrieve connection details for the database"""
    try:
        # Get database details
        db = w.database.get(id=database_id)
        
        # Get connection string
        connection_string = w.database.get_connection_string(id=database_id)
        
        details = {
            "database_id": db.id,
            "database_name": db.name,
            "state": db.state,
            "connection_string": connection_string,
            "instance_type": INSTANCE_TYPE,
            "ssl_enabled": True
        }
        
        return details
        
    except Exception as e:
        print(f"✗ Failed to retrieve connection details: {str(e)}")
        raise

# Get connection details
connection_details = get_connection_details(ready_database.id)

print("\n" + "="*60)
print("CONNECTION DETAILS")
print("="*60)
for key, value in connection_details.items():
    if key != "connection_string":  # Don't print full connection string
        print(f"{key:20s}: {value}")
print("="*60)


## Save Configuration


In [ ]:
# Save configuration as notebook widget for easy access
dbutils.widgets.text("database_id", connection_details["database_id"], "Database ID")
dbutils.widgets.text("database_name", connection_details["database_name"], "Database Name")

print(f"\n✓ Configuration saved as notebook widgets")
print(f"✓ Database ID: {connection_details['database_id']}")
print(f"✓ Database Name: {connection_details['database_name']}")


## Summary


In [ ]:
print("\n" + "="*60)
print("SETUP COMPLETE")
print("="*60)
print(f"✓ Lakebase PostgreSQL database created: {DATABASE_NAME}")
print(f"✓ Database ID: {connection_details['database_id']}")
print(f"✓ State: {connection_details['state']}")
print(f"✓ Instance Type: {INSTANCE_TYPE}")
print(f"✓ SSL: Enabled")
print(f"\nNext Steps:")
print(f"  1. Run notebook 02-create-schema.ipynb to create database schema")
print(f"  2. Insert sample expense data")
print(f"  3. Set up sync pipeline to Lakehouse")
print("="*60)
